# Introdução: 
O preço das passagens aéreas muda o tempo todo e pode variar bastante, o faz esse tema ser interessante para tentar predizer o preço através de outros fatores. No nosso Dataset temos informações sobre vôos entre diversas cidades da Índia, sendo os preditores: a companhia aérea, o trajeto entre as cidades de origem e destino, a classe escolhida, o número de escalas, a duração do voo e a antecedência com que a passagem é comprada. O desafio é entender como cada um desses aspectos pesa na hora de definir quanto o passageiro vai pagar no final.



# Análise Exploratória: 


## Setup

In [ ]:
# Importando bibliotecas

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, OneHotEncoder
import xgboost as xgb
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from pygam import s, f, LinearGAM
from sklearn.ensemble import RandomForestRegressor
from sklearn.compose import ColumnTransformer
from sklearn.cluster import KMeans
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import silhouette_score
from sklearn.ensemble import IsolationForest
from sklearn.linear_model import RidgeCV
from sklearn.model_selection import cross_validate
from sklearn.model_selection import KFold, cross_val_score
from sklearn.linear_model import LassoCV
from sklearn import linear_model
from pygam import LinearGAM, s, l
from sklearn import metrics
from sklearn.linear_model import LinearRegression
from numpy.linalg import norm
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
import numpy as np
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules
import altair as alt
import vegafusion
import geopandas as gpd
import geodatasets
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable, get_cmap



### Importando base de dados (Flight)

In [ ]:
df_flight = pd.read_csv('data/flight.csv')
df_flight.head()

## Limpeza e Qualidade dos dados

In [ ]:
# Removendo a primeira coluna, pois não acrescenta na análise

df_flight = df_flight.drop('Unnamed: 0',axis=1)

A primeira coluna, Unnamed: 0, é um índice redundante que não adiciona valor preditivo ou contextual à nossa análise. Para garantir a limpeza e foco do dataset, optamos por remover essa coluna antes de prosseguir com a análise descritiva e visual.

In [ ]:
# Informações
df_flight.info()

# Verificar valores nulos
df_flight.isnull().sum()

Aqui podemos observar que não temos valores nulos a serem tratados.


### Colunas

As colunas existentes no dataset escolhido são:

1. **airline** – Companhia aérea responsável pelo voo.

2. **flight** – Código ou número identificador do voo.

3. **source_city** – Cidade de origem de onde o voo parte.

4. **departure_time** – Faixa de horário da decolagem (por exemplo: manhã, tarde, noite).

5. **stops** – Número de paradas intermediárias (ex: zero, one, two_or_more).

6. **arrival_time** – Faixa de horário de chegada ao destino.

7. **destination_city** – Cidade de destino para onde o voo está indo.

8. **class** – Classe da passagem (por exemplo: Economy, Business).

9. **duration** – Duração total do voo em horas.

10. **days_left** – Quantidade de dias restantes até a data do voo no momento da coleta dos dados.

11. **price** – Preço da passagem aérea.



Com a estrutura e qualidade dos dados verificadas, o foco se volta para as variáveis que servirão como preditores do preço da passagem (price). A compreensão de cada uma é crucial para guiar as visualizações e o futuro modelo de predição.



## Análise unidimensional



Para ter uma visão inicial da distribuição e das tendências centrais das variáveis numéricas, geramos o resumo estatístico que detalha a contagem, média, desvio padrão, valores mínimos/máximos e os quartis.

In [ ]:
df_flight.describe()

A análise do resumo estatístico mostra:

* duration (Duração): Os voos têm uma duração média de 12,2 horas, com um desvio padrão alto, indicando uma grande variabilidade no tempo de viagem, provavelmente devido à inclusão de paradas.

* days_left (Antecedência): As passagens são compradas, em média, com 26 dias de antecedência. O desvio padrão é de 13,56 o que é bem próximo do que seria uma distribuição uniforme nesse intervalo, mas ainda não podemos afirmar que é. O que tiramos de conclusão é que são dados relativamente bem espalhados também.

* price (Preço): O preço médio é de aproximadamente 20.889 ($), com um desvio padrão que sugere uma dispersão considerável e a possível presença de outliers, o que será melhor visualizado nos boxplots.

In [ ]:
# Pela quantidade de observações, é necessário desabilitar o máximo de colunas padrão do Altair (5000)

alt.data_transformers.disable_max_rows()

##### Histogramas

A seguir, visualizamos a distribuição de frequência das variáveis numéricas através de histogramas no Altair, uma técnica que permite identificar a forma da distribuição, o centro e a dispersão dos dados. É particularmente útil para verificar se as variáveis se aproximam de uma distribuição normal ou se há assimetrias.

In [ ]:
num_cols = ["duration", "days_left", "price"]

charts = [
    alt.Chart(df_flight).mark_bar().encode(
        alt.X(col, type="quantitative", bin=alt.BinParams(maxbins=15), title=col),
        alt.Y("count()", type="quantitative", title="Contagem de ocorrências")
    ).properties(width=150, height=150)
    for col in num_cols
]

histogramas_num = alt.hconcat(*charts).properties(
    title="Distribuição de variáveis numéricas"
)

histogramas_num


Os histogramas confirmam a dispersão observada:

* duration: Mostra uma distribuição aparentemente normal.

* days_left: Apresenta uma distribuição relativamente plana (uniforme), o que corrobora a análise descritiva sobre a homogeneidade na antecedência de compra.

* price: Exibe uma distribuição positivamente assimétrica (à direita), com a maioria dos preços concentrada em valores mais baixos e uma cauda longa para valores mais altos, indicando a presença de voos premium ou de longa duração.

##### Histogramas das variáveis categóricas

A análise de variáveis categóricas continua com histogramas de frequência, que nos permitem avaliar a distribuição das observações entre as diferentes categorias de cada preditor, como airline, class e source_city.

In [ ]:
cat_cols = [
    "airline", "source_city", "departure_time", "stops",
    "arrival_time", "destination_city", "class"
]
charts_cat = [
    alt.Chart(df_flight).mark_bar().encode(
        x=alt.X(col, type="nominal", title=col, sort='-y'),
        y=alt.Y("count()", type="quantitative", title="Contagem de ocorrências"),
        tooltip=[col, "count()"]
    ).properties(width=150, height=150)
    for col in cat_cols
]

# organiza os gráficos em uma grade
histogramas_cat = alt.hconcat(*charts_cat[:4]).resolve_scale(y='independent')
if len(cat_cols) > 4:
    row2 = alt.hconcat(*charts_cat[4:]).resolve_scale(y='independent')
    histogramas_cat = alt.vconcat(histogramas_cat, row2)

histogramas_cat = histogramas_cat.properties(
    title="Distribuição de variáveis categóricas"
)

histogramas_cat


Os histogramas categóricos revelam a proporção de dados por categoria, sendo útil para identificar desequilíbrios que podem influenciar modelos preditivos:

* airline: Vistara é a companhia aérea com o maior número de voos no dataset.

* class: Há um claro desequilíbrio entre as classes, com Economy dominando a maioria das observações.

* source_city e destination_city: Delhi e Mumbai são as cidades mais representadas, tanto como origem quanto como destino.

* stops: É possível observar uma grande quantidade de ocorrencias de uma parada sendo mais de 75% dos dados.

##### Boxplots

Para complementar a análise univariada das variáveis numéricas, utilizamos boxplots. Estes gráficos são ideais para identificar a dispersão, a simetria dos dados e, principalmente, a presença e extensão de outliers, que podem impactar a robustez do modelo.

In [ ]:
col_titles = {
    "duration": "Duração do voo (horas)",
    "days_left": "Dias restantes até o voo",
    "price": "Preço da passagem (R$)"
}


plt.figure(figsize=(15, 4))

# cria um boxplot horizontal para cada variável numérica
for i, col in enumerate(num_cols):
    plt.subplot(1, 3, i+1) 
    sns.boxplot(x=df_flight[col])  
    plt.title(f"Boxplot de {col_titles[col]}")
    plt.grid(axis="x", linestyle="--", alpha=0.6)

plt.tight_layout()
plt.show()

Os boxplots fornecem informações visuais sobre a dispersão:

* duration: A mediana está em torno de 10 horas, mas há uma grande quantidade de outliers na cauda superior, confirmando a alta variabilidade e a concentração de voos de longa duração.

* days_left: A distribuição é bem centralizada e simétrica, com poucos outliers extremos, reforçando a uniformidade já percebida.

* price: A distribuição está concentrada na parte inferior, com uma extensa cauda de outliers (preços muito altos), o que é esperado para passagens de classe Business ou compras de última hora. Este é um indicativo de que o preço é a variável mais sensível a valores extremos no dataset.

## Análise Bidimensional


A análise bidimensional é crucial para entender as relações e a correlação entre os preditores e a variável alvo, o price. Começamos examinando a relação entre as variáveis numéricas.

Vamos começar por um mapa de calor das correlações entre as variáveis numéricas:

In [ ]:
num_cols = ["duration", "days_left", "price"]

plt.figure(figsize=(10, 6))
sns.heatmap(df_flight[num_cols].corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Mapa de Calor - Correlação entre variáveis numéricas')
plt.show()

O mapa de calor de correlação revela a seguinte relação:
* duration vs. price: Apresentam uma correlação moderada-positiva ($0.20$). Embora a duração do voo (incluindo escalas) não seja o principal fator, voos mais longos tendem a ser um pouco mais caros.


Agora prosseguimos para tentar ver as relações visualmente através de scaterplots:

In [ ]:
sns.pairplot(
    df_flight[num_cols],
    diag_kind="kde",          
    corner=True,              
    plot_kws={'alpha': 0.6, 's': 40, 'edgecolor': 'none'},
    diag_kws={'fill': True, 'color': '#4C72B0'}
)

plt.suptitle("Scatter Plot das Variáveis Numéricas", y=1.02, fontsize=14)
plt.show()


Esse plot nos permite reforçar o que percebemos sobre a variavel days_left sobre sua proximidade de uma distribuição uniforme. Além disso observamos novamente que tanto duration quanto price tem um pico na parte esquerda do gráfico mas com uma cauda bastante longa que acaba aumentando a média e distanciando-a da mediana. Quanto aos graficos de uas váriaveis não é possível observar nada muito concreto apenas algumas nuevns de pontos.

O pairplot anterior com todos os dados não permitiu identificar padrões claros. Para melhorar a visualização das interações, geramos um gráfico de dispersão (scatter plot) com uma amostra de 1.000 observações, colorido pela airline (companhia aérea). Embora ainda haja dispersão, este plot com amostra nos dá um vislumbre das faixas de preço e duração por companhia.

In [ ]:
df_sample = df_flight.sample(n=1000, random_state=42)

# --- Gerar o pairplot com estilo limpo ---
sns.pairplot(
    df_sample,
    vars=num_cols,
    hue="airline",
    diag_kind="kde",
    corner=True,
    plot_kws={'alpha': 0.6, 's': 35, 'edgecolor': 'none'}
)

plt.suptitle("Scatter Plot das Variáveis Numéricas (Amostra de 1.000 voos)", y=1.02, fontsize=14)
plt.show()

A partir desse gráfico, é interessante buscar o padrão de preços por companhia aérea. O boxplot a seguir visualiza a distribuição de preços para cada companhia, permitindo comparar não apenas as medianas, mas também a dispersão e a presença de outliers.

In [ ]:
plt.figure(figsize=(10, 6))
sns.boxplot(data=df_flight, x='airline', y='price',color= 'red')
plt.title('Distribuição do Preço por Companhia Aérea')
plt.xticks(rotation=45)
plt.show()


O boxplot de preço por Companhia Aérea revela uma diferença acentuada nas políticas de preço:

* Vistara é a companhia com a maior mediana de preços e também com a maior dispersão (amplitude interquartil).

* AirAsia e SpiceJet apresentam as menores medianas, confirmando a percepção de que são companhias de custo mais baixo no dataset.

* A presença de outliers na cauda superior para quase todas as companhias é evidente, indicando a venda de passagens Business ou rotas de alta demanda.

A hora do dia em que um voo parte ou chega pode ter um impacto direto no preço, devido à demanda e conveniência. Analisamos o preço médio por horário de partida (departure_time) e horário de chegada (arrival_time).

In [ ]:
departure_price = df_flight.groupby('departure_time')['price'].mean().reset_index()
departure_price

In [ ]:
arrival_price = df_flight.groupby('arrival_time')['price'].mean().reset_index()
arrival_price

Aqui podemos perceber que voos que partem muito tarde tendem a ter preços expressivamente mais baixos e os que chegam nessa faixa de horário também tendem a ser mais baratos.


In [ ]:

sns.set(style="whitegrid")

fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharey=True)

# --- Partida ---
sns.boxplot(
    data=df_flight,
    x='departure_time',
    y='price',
    color=sns.color_palette("viridis", n_colors=1)[0],  # única cor
    ax=axes[0]
)
axes[0].set_title('Preço por Horário de Partida')
axes[0].set_xlabel('Horário de Partida')
axes[0].set_ylabel('Preço')
axes[0].grid(True, linestyle='--', alpha=0.3)

# --- Chegada ---
sns.boxplot(
    data=df_flight,
    x='arrival_time',
    y='price',
    color=sns.color_palette("magma", n_colors=1)[0],  # única cor
    ax=axes[1]
)
axes[1].set_title('Preço por Horário de Chegada')
axes[1].set_xlabel('Horário de Chegada')
axes[1].set_ylabel('')
axes[1].grid(True, linestyle='--', alpha=0.3)

plt.tight_layout()
plt.show()


Agora vamos tentar juntar essas informações em uma única visualização. O preço é influenciado pela combinação dos horários de partida e chegada. O gráfico de linha ilustra essa interação, mostrando como o preço médio de chegada varia, segmentado pelo horário de partida. Isso ajuda a entender o valor cobrado pela conveniência de rotas específicas

In [ ]:
sns.set_theme(style="whitegrid")

g = sns.relplot(
    data=df_flight,
    x="arrival_time",
    y="price",
    col="departure_time",
    kind="line",
    marker="o",
    linewidth=2,
    height=4,
    aspect=1,
    palette="viridis"
)

# Ajustes estéticos
g.set_titles(col_template="Partida: {col_name}")
g.set_axis_labels("Horário de Chegada", "Preço Médio")
g.fig.suptitle("💸 Relação entre Preço e Horário de Chegada por Horário de Partida", 
               fontsize=14, fontweight="bold", y=1.05)

# Rotação dos rótulos do eixo X
for ax in g.axes.flatten():
    ax.tick_params(axis='x', rotation=45)

plt.show()

O gráfico de linha confirma a complexidade da relação Preço x Horário de Voo:

* Partidas 'Night' (Noite) e 'Morning' (Manhã): Estas rotas tendem a ter preços médios mais altos em quase todos os horários de chegada, sugerindo que o horário de partida é um forte preditor de preço.

* Voos 'madrugadores' ('Early_Morning'): As linhas para voos de manhã cedo (seja na partida ou na chegada) tendem a mostrar picos de preço em horários de chegada mais 'nobres', como 'Morning' (Manhã) e 'Evening' (Noite), o que pode indicar rotas com escalas e longa duração.

* 'Late_Night' (Madrugada): Rotas com partida ou chegada de madrugada tendem a ser consistentemente mais baratas, com poucas exceções.

Identificadas as principais cidades no dataset, a análise prossegue para um dos preditores de preço mais intuitivos: a rota (cidade de origem e destino) e, implicitamente, a distância percorrida. Analisar como o preço varia entre as cidades é o próximo passo.

In [ ]:
cities = ['Delhi', 'Mumbai', 'Bangalore', 'Kolkata', 'Hyderabad', 'Chennai']


df_filtered = df_flight[
    df_flight['source_city'].isin(cities) & df_flight['destination_city'].isin(cities)
]


price_matrix = (
    df_filtered.groupby(['source_city', 'destination_city'])['price']
    .mean()
    .unstack()
    .reindex(index=cities, columns=cities)
)


np.fill_diagonal(price_matrix.values, np.nan)


price_matrix.style.background_gradient(cmap="RdYlGn_r", axis=None)\
    .set_caption("Preço Médio das Passagens entre Principais Cidades (₹)")\
    .format("{:.0f}")

A matriz de calor de preços médios revela as rotas mais e menos caras:

* Rotas Mais Caras (Tons Mais Escuros): As passagens de Chennai para Bangalore ($25.082) e de Kolkata para Chennai ($23.660) estão entre as mais caras.

* Rotas Mais Baratas (Tons Mais Claros/Verdes): As rotas de Hyderabad para Delhi ($17.244) e de Delhi para Hyderabad ($17.347) apresentam preços médios mais baixos, sugerindo que a localização geográfica não é o único fator de custo.

Para melhor visualização, vamos analisar também os boxplots das cidades de partida e chegada:

In [ ]:


sns.set(style="whitegrid")

fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharey=True)

# --- Paletas discretas ---
palette_source = sns.color_palette("viridis", n_colors=df_flight['source_city'].nunique())
palette_dest = sns.color_palette("magma", n_colors=df_flight['destination_city'].nunique())

# --- Boxplot: Cidade de Origem ---
sns.boxplot(
    data=df_flight,
    x='source_city',
    y='price',
    hue='source_city',
    palette=palette_source,
    legend=False,
    ax=axes[0]
)
axes[0].set_title('Preço por Cidade de Origem', fontsize=13)
axes[0].set_xlabel('Cidade de Origem')
axes[0].set_ylabel('Preço')
axes[0].tick_params(axis='x', rotation=45)
axes[0].grid(True, linestyle='--', alpha=0.3)

# --- Boxplot: Cidade de Destino ---
sns.boxplot(
    data=df_flight,
    x='destination_city',
    y='price',
    hue='destination_city',
    palette=palette_dest,
    legend=False,
    ax=axes[1]
)
axes[1].set_title('Preço por Cidade de Destino', fontsize=13)
axes[1].set_xlabel('Cidade de Destino')
axes[1].set_ylabel('')
axes[1].tick_params(axis='x', rotation=45)
axes[1].grid(True, linestyle='--', alpha=0.3)

plt.tight_layout()
plt.show()


* Origem: Voos partindo de Chennai e Kolkata tendem a ter as medianas de preço mais altas. Em contraste, Hyderabad apresenta a mediana mais baixa, sugerindo que as passagens aéreas originadas nesta cidade tendem a ser mais acessíveis.

* Destino: Voos com destino a Chennai e Bangalore mostram medianas elevadas, indicando que a chegada a esses centros urbanos pode estar associada a custos mais altos.

* Conclusão: A variação do preço é altamente sensível à cidade, tanto na partida quanto na chegada. No entanto, a grande dispersão (outliers na cauda superior) em todas as cidades sugere que o preço final depende crucialmente da combinação de fatores, como a classe da passagem e a companhia aérea, que superam o impacto isolado da localização geográfica.

In [ ]:

city_coords = {
    'Delhi': (28.6139, 77.2090),
    'Mumbai': (19.0760, 72.8777),
    'Bangalore': (12.9716, 77.5946),
    'Kolkata': (22.5726, 88.3639),
    'Hyderabad': (17.3850, 78.4867),
    'Chennai': (13.0827, 80.2707)
}

def haversine_distance(city1, city2):
    from math import radians, sin, cos, sqrt, atan2
    if city1 not in city_coords or city2 not in city_coords:
        return np.nan
    lat1, lon1 = city_coords[city1]
    lat2, lon2 = city_coords[city2]
    R = 6371
    dlat, dlon = radians(lat2 - lat1), radians(lon2 - lon1)
    a = sin(dlat/2)**2 + cos(radians(lat1))*cos(radians(lat2))*sin(dlon/2)**2
    return R * 2 * atan2(sqrt(a), sqrt(1 - a))


if 'distance' not in df_flight.columns:
    df_flight['distance'] = df_flight.apply(
        lambda row: haversine_distance(row['source_city'], row['destination_city']),
        axis=1
    )


route_stats = (
    df_flight.groupby(['source_city', 'destination_city'])
    .agg({'price': 'mean', 'distance': 'mean'})
    .reset_index()
)
route_stats['src_lat'] = route_stats['source_city'].map(lambda x: city_coords.get(x, (None, None))[0])
route_stats['src_lon'] = route_stats['source_city'].map(lambda x: city_coords.get(x, (None, None))[1])
route_stats['dst_lat'] = route_stats['destination_city'].map(lambda x: city_coords.get(x, (None, None))[0])
route_stats['dst_lon'] = route_stats['destination_city'].map(lambda x: city_coords.get(x, (None, None))[1])


world_path = geodatasets.get_path("naturalearth.land")
world = gpd.read_file(world_path)
india = world.cx[67:98, 6:37]


fig, ax = plt.subplots(figsize=(8, 8))
india.plot(ax=ax, color="#f8f9fa", edgecolor="lightgray", linewidth=0.6)


cmap = get_cmap("RdYlGn_r")  
norm = Normalize(vmin=route_stats['price'].min(), vmax=route_stats['price'].max())


for _, row in route_stats.iterrows():
    color = cmap(norm(row['price']))
    plt.plot(
        [row['src_lon'], row['dst_lon']],
        [row['src_lat'], row['dst_lat']],
        color=color,
        linewidth=2,
        alpha=0.8,
        zorder=2
    )


for city, (lat, lon) in city_coords.items():
    plt.scatter(lon, lat, s=100, color='white', edgecolors='black', linewidth=1.5, zorder=3)
    plt.text(lon + 0.4, lat + 0.3, city, fontsize=9, fontweight='bold', color='#1d3557')


sm = ScalarMappable(norm=norm, cmap=cmap)
cbar = plt.colorbar(sm, ax=ax, fraction=0.03, pad=0.02)
cbar.set_label('Preço médio da passagem (₹)', fontsize=10)
cbar.ax.tick_params(labelsize=8)


plt.xlim(67, 98)
plt.ylim(6, 33)
plt.title("Mapa da Índia Rotas Aéreas com Heatmap de Preço Médio", fontsize=13, fontweight='bold', pad=15)
plt.axis('off')
plt.tight_layout()
plt.show()


Para verificar a hipótese da distância ser o principal preditor, visualizamos as rotas no mapa real da Índia, onde a cor de cada linha representa o preço médio (vermelho/escuro = mais caro; verde/claro = mais barato).

O mapa desmente a correlação direta entre distância física e preço. Rotas mais curtas podem ser mais caras e vice-versa. Rotas como Chennai para Bangalore (geograficamente próximas) são mais caras do que rotas mais longas, como Hyderabad para Delhi. Isso reforça a ideia de que a demanda do mercado, a competição entre companhias e a qualidade do serviço (classe e número de escalas) são preditores mais importantes do que a distância.

Após analisarmos as correlações numéricas, focamos na relação crítica entre duração do voo (duration) e o preço (price). No entanto, essa relação é fortemente mediada pela companhia aérea.

O gráfico de dispersão a seguir, utilizando uma amostra (df_sample) para maior clareza, nos permite visualizar como o preço se distribui em função da duração do voo, com as cores diferenciando as companhias aéreas. Isso é essencial para identificar se certas empresas dominam as faixas de voos mais longos e caros, revelando a combinação de fatores que impulsiona o custo final.

In [ ]:
sns.scatterplot(
    data=df_sample, x='duration', y='price', hue='airline', alpha=0.6
)
plt.title("Preço vs Duração do Voo por Companhia Aérea")
plt.show()

O scatter plot (acima) reforçou que a Companhia Aérea (airline) é um forte preditor de preço, sendo as companhias de maior custo (como Vistara) associadas a voos com maior duração e faixas de preço elevadas.

Em seguida, focaremos em um fator estrutural que está intimamente ligado tanto à duração quanto à conveniência: o Número de Paradas (stops). A lógica aqui é que menos paradas implicam um voo mais rápido e, teoricamente, mais conveniente, o que deve influenciar o preço. O boxplot a seguir irá quantificar como o preço médio e sua distribuição variam para voos diretos (zero), com uma parada (one) ou com múltiplas paradas (two_or_more).

In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(
    data=df_flight,
    x='stops',
    y='price'
)
plt.title('Preço da Passagem por Número de Paradas', fontsize=13)
plt.xlabel('Número de Paradas')
plt.ylabel('Preço (₹)')
plt.grid(True, linestyle='--', alpha=0.3)
plt.show()


Como a análise anterior demonstrou que voos com uma parada (one stop) apresentam a mediana de preço mais alta, é crucial examinar essa variável em um contexto bidimensional. O gráfico de dispersão a seguir cruza Duração do Voo (duration), Preço (price) e Número de Paradas (stops) simultaneamente.

Esta visualização nos permitirá confirmar se a categoria 'one stop' domina os voos mais longos e caros, atuando, na prática, como um proxy para as passagens de Classe Executiva ou Primeira Classe, que frequentemente incluem uma escala em rotas internacionais ou intercontinentais.

In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(
    data=df_sample,
    x='duration',
    y='price',
    hue='stops',
    palette='coolwarm',
    alpha=0.6
)
plt.title('Preço vs Duração, colorido pelo Número de Paradas', fontsize=13)
plt.xlabel('Duração (horas)')
plt.ylabel('Preço (₹)')
plt.grid(True, linestyle='--', alpha=0.3)
plt.show()



O scatter plot (acima) confirma de forma visualmente poderosa a hipótese levantada:

* Escalas 'one' (laranja/vermelho): Estes voos dominam claramente a faixa de preço mais alta, estendendo-se por quase todo o espectro de duração, inclusive nas viagens mais longas. Isso sugere fortemente que esta categoria absorve a maior parte das passagens de alto valor (como as de classe Business ou First).

* Escalas 'zero' (azul): Embora sejam voos diretos e rápidos, estão restritos a uma faixa de preço muito mais baixa, indicando que são predominantemente passagens de Classe Econômica em rotas regionais.

A partir disso, concluímos que o Número de Paradas é um preditor fortíssimo, mas seu efeito no preço é mais um reflexo da Classe da Passagem e da duração total do que da inconveniência da escala em si.

Até o momento, a análise exploratória permitiu compreender melhor a estrutura e a qualidade do conjunto de dados, destacando as variáveis mais relevantes e suas características gerais. Observou-se a presença de fatores que podem influenciar o preço das passagens, como a companhia aérea, o número de paradas, a duração do voo e a antecedência da compra. Esses padrões iniciais ajudam a delinear hipóteses sobre como cada atributo pode contribuir para a formação do valor final. A partir dessas observações, espera-se que as próximas etapas, envolvendo a regressão e outros modelos preditivos, aprofundem a relação entre as variáveis e permitam identificar quais fatores exercem maior impacto sobre o preço das passagens.

In [ ]:
categorical_features = ['airline', 'source_city', 'departure_time', 'stops',
                        'arrival_time', 'destination_city', 'class']
numerical_features = ['duration', 'days_left']


## Normalizando os dados

Após isso, aplicaremos transformações às colunas com objetivo de normalizar os dados, isso é necessário pois o k-means calcula distancias e ter alguma variavel numericamente muito maior que as outras ou muito menor pode causar problemas.

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(), categorical_features)
    ]
)
X_prepared = preprocessor.fit_transform(df_flight)

Agora calcularemos a inercia para o k-means utilizando k de 1 até 11, isso serve para escolhermos qual seria o melhor k para usar, a inercia aqui significa a soma das distâncias quadráticas das amostras ao centro do cluster mais próximo.

In [ ]:
inertia = []
K_range = range(1, 11) # Testaremos K de 1 a 10

In [ ]:
for k in K_range:
    # n_init='auto' é o padrão moderno para evitar warnings futuros
    kmeans_model = KMeans(n_clusters=k, random_state=42, n_init='auto')
    kmeans_model.fit(X_prepared)
    inertia.append(kmeans_model.inertia_)

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(K_range, inertia, marker='o', linestyle='--')
plt.xlabel('Número de Clusters (K)')
plt.ylabel('Inércia')
plt.title('Método do Cotovelo para K Ótimo')
plt.xticks(K_range)
plt.grid(True)
plt.show()

Uma possibilidade para escolher o k ideal é analisar o "cotovelo" do gráfico, isso é, onde a inercia ainda decresce bastante e após esse valor para de diminuir tanto, nesse caso podemos observar que é o k = 4.

In [ ]:
optimal_k = 4

kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init='auto')
clusters = kmeans.fit_predict(X_prepared)

In [ ]:
df_flight['cluster'] = clusters


print(f"\n### Análise dos {optimal_k} Clusters Encontrados (Médias das Variáveis Numéricas) ###\n")
cluster_analysis_numeric = df_flight.groupby('cluster')[numerical_features].mean().round(2)
print(cluster_analysis_numeric)

for feature in categorical_features:
    print(f"\n--- Distribuição de '{feature}' por Cluster ---")
    print(pd.crosstab(df_flight['cluster'], df_flight[feature]))

Fazendo uma rápida analise dos clusters podemos perceber observando as variaveis duration e days_left que o cluster 0 tende a ter voos mais longos comprados em datas mais próximas à data da viagem. Já o cluster 3 também tem duração média longa mas com compras mais antecipadas. O cluster 1 tem voos curtos comprados com grande antecedencia e por fim o cluster 2 tende a ter voos curtos comprados em cima da hora. Ademais é possivel notar que o cluster 1 e 2 tem maior presença de classe economica que o 0 e 3. No 0 e 3 temos uma divisao que se aproxima de 50/50 enquanto no 1 e 2 a classe economica domina fortemente.

**Preparação dos Dados para Regressão Linear**

Preparamos os dados para que possam ser utilizados adequadamente pelo modelo de regressão. O conjuto de dados original contém variáveis categóricas (*cidade de origem, companhia aérea etc*), utilizamos o método:  

```
pd.get_dummies()
```

Cada categoria é transformado numa nova coluna binária (ou preditor binário), permitindo que o modelo interprete a informação qualitativa. Para evitar o problema de multicolinearidade utilizamos o parâmetro:

```
drop_first = True
```

Em seguida, definimos as variáveis:



*   **x:** conjnto de atributos preditores. Removemos as colunas não utilizadas no treinamento: flight, price e cluster;
*   **y:** variável alvo: preço das passagens aéreas.

Os dados são dividos em conjuntos de **treinamento e teste** na proporção de 80% para treino e 20% para teste. 

In [ ]:
df_encoded = pd.get_dummies(df_flight, columns=['airline', 'source_city', 'departure_time', 'stops', 'arrival_time', 'destination_city', 'class', 'cluster'], drop_first=True)

X = df_encoded.drop(['flight', 'price'], axis=1)
y = df_encoded['price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

**Padronização das Variáveis**

Padronização das variáveis numéricas utilizando:

```
StandardScaler()
```

Esse procedimento garante que todas as variáveis estejam na mesma escala, para não haver o problema de atributos com valores maiores dominem o processo de aprendizagem. 

```
fit_transform()
transform()
```
Métodos aplicados somente sobre o conjunto de treinamento e teste, respectivamente. 

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

# Aplicamos a mesma transformação nos dados de teste
X_test_scaled = scaler.transform(X_test)

# REGRESSÃO LINEAR

**Modelo de Regressão Linear**

A regressão linear é um algoritmo de **aprendizagem supervisionada** usado quando a variável alvo é um número real contínuo (no caso, o *preço da passagem aérea*). Ela estabelece a relação entre a variável dependente **y** e uma ou mais variáveis **x** usando a linha de melhor ajuste. Utilizamos o método:

```
fit()
```
Para realizar a estimação dos parâmetros através do **método dos mínimos quadrados ordinários (OLS - *Ordinary Least Squares*)**.  

O modelo assume uma relação linear da forma:

$$
\hat{y} = \theta_0 + \theta_1 x_1 + \theta_2 x_2 + \cdots + \theta_p x_p
$$

onde:


*   $\hat{y}$ representa o valor estimado do preço; 
*   $\theta_{0}$ é o intercepto;
*   $\theta_{i}$ são os coeficientes de regressão associados a cada variável $x_{i}$.

In [ ]:
model = LinearRegression()
model.fit(X_train_scaled, y_train)

**Avaliação do Modelo de Regressão Linear**

Avaliamos o desempenho tanto no conjunto de treino (verificar o quanto o modelo aprendeu os padrões dos dados) e teste (medir capacidade de generalização).

Foram utilizadas as seguinte métricas:

*   **$R^2$ (Coeficiente de Determinação):** Mede proporção da variabilidade do preço das passagens explicada pelo modelo. Valor entre 0 e 1, quando mais próximo de 1 melhor o ajuste:

$$
R^2 = 1 - \frac{\sum (y_i - \hat{y_i})^2}{\sum (y_i - \bar{y})^2}
$$

*   **MAE (Mean Absolute Error):** Erro médio absoluto entre os valores previstos e os reais, em unidades monetárias:

$$
MAE = \frac{1}{n} \sum |y_i - \hat{y_i}|
$$

*   **RMSE (Root Mean Squared Error):** Erro quadrático médio, penaliza mais grandes discrepâncias:

$$
RMSE = \sqrt{\frac{1}{n} \sum (y_i - \hat{y_i})^2}
$$

Essa métricas permitem avaliar tanto o **ajuste do modelo** (via $R^2$) quanto a **magnitude dos erros de previsão** (via $MAE$ e $RMSE$). 

In [ ]:
y_pred_train = model.predict(X_train_scaled)
y_pred_test = model.predict(X_test_scaled)

r2_train = metrics.r2_score(y_train, y_pred_train)
r2_test = metrics.r2_score(y_test, y_pred_test)

mae_train = metrics.mean_absolute_error(y_train, y_pred_train)
mae_test = metrics.mean_absolute_error(y_test, y_pred_test)

rmse_train = np.sqrt(metrics.mean_squared_error(y_train, y_pred_train))
rmse_test = np.sqrt(metrics.mean_squared_error(y_test, y_pred_test))

print("RESULTADOS OLS (Treino vs Teste)")
print(f"R² Treino: {r2_train:.4f}")
print(f"R² Teste:  {r2_test:.4f}")
print(f"MAE Treino: {mae_train:,.2f}")
print(f"MAE Teste:  {mae_test:,.2f}")
print(f"RMSE Treino: {rmse_train:,.2f}")
print(f"RMSE Teste:  {rmse_test:,.2f}")

Os valores de $R^2$ para treino (0.9115) e (0.9113) são muito próximos, mostrando que o modelo não sofreu overfitting e consegue explicar cerca de 91% da variação no preço das passagens. 

Os erros médios ($MAE ≈ 4.500$ e $RMSE ≈ 6.700$) são consistentes entre treino e teste: sugere modelo estável e robusto nas previsões. 

In [ ]:
print("Coeficientes do Modelo")
coeficientes = pd.DataFrame(model.coef_, X.columns, columns=['Coeficiente'])
print(coeficientes)

**a) O preço varia de acordo com a companhia aérea?**

O preço varia sim entre as companhias aéreas. Companhias como Vistara e a Indigo apresentam tarifas médias mais altas, enquanto Air India e Spicejet têm impacto menor. 

**b) Como o preço é afetado quando as passagens são compradas com apenas 1 ou 2 dias de antecedência?**

O coeficiente de **days_left** (-1769.9) mostra que quanto menor o número de dias até a viagem, maior o preço da passagem. Comprar com antecedência reduz o custo, mas adquirir o bilhete próximo a data do voo encarece o valor. 

**c) O preço da passagem muda de acordo com o horário de partida e chegada?**

Tanto o horário de partida (**departure_time**) quanto o de chegada (**arrival_time**) afetam o preço. Voos que partem pela manhã ou a noite apresentam coeficientes positivos sugerem que horários mais convenientes ou de demanda alta tendem a ter preços mais altos. 

**d) Como o preço muda com a alteração da origem e do destino?**

Os coeficientes de **source_city** e **destination_city** indicam diferenças regionais nos preços médios. Voos que envolvem Kolkata têm coeficientes positivos, sugerindo valores mais altos enquanto Delhi e Hyderabad valores mais baixos. 

**e) Como o preço da passagem varia entre a Classe Econômica e a Classe Executiva?**

O coeficiente da variável **class_Economy** (-20804.96) mostra que as passagens da classe enconômica são ₹20.800 mais baratas do que as da classe executiva. A classe de serviço é o principal determinante do valor da passagem aérea. 

**Validação Cruzada (K-Fold)**

Para validar a consistência do modelo, aplicou-se uma validação cruzada com 10 partições (K=10). 

Em cada uma das 10 iterações, o modelo é treinado em 90% dos dados e testado nos 10% restantes, de forma aleatória e balanceada.

In [ ]:
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LinearRegression())
])

kf = KFold(n_splits=10, shuffle=True, random_state=42)

resultados = cross_validate(
    pipeline,
    X,
    y,
    cv=kf,
    scoring=['r2', 'neg_mean_absolute_error', 'neg_root_mean_squared_error'],
    return_estimator=False
)

print("\n--- Resultados da Avaliação com K-Fold ---")
print("R² médio:", resultados['test_r2'].mean().round(4))
print("MAE médio:", (-resultados['test_neg_mean_absolute_error'].mean()).round(4))
print("RMSE médio:", (-resultados['test_neg_root_mean_squared_error'].mean()).round(4))


Esses valores confirmam que o modelo apresenta capacidade explicativa e boa generalização, já que o desempenho médio é muito semelhante ao observado na avaliação simples de treino e teste. Isso indica estabilidade e consistência na previsão dos preços das passagens aéreas.

# RIDGE

O modelo **Ridge** adiciona uma penalização L2 na Regressão Lienar, reduzindo o tamanho dos coeficientes para evitar *overfitting*. Função custo para ser minimizada:

$$
\sum_{i=1}^{n} (y_i - \hat{y_i})^2 + \alpha \sum_{j=1}^{p} \theta_j^{2}
$$

O parâmetro $\alpha $ controla a intensidade da penalização.


**Usando K-Folds**

Foram testados diversos valores do parâmetro de regularização $\alpha$ para o modelo Ridge. O objetivo era verificar se o acréscimo da penalização L2 melhoraria o desempenho do modelo em relação à Regressão Linear simples.

In [ ]:
alphas = [0.1, 1, 10, 100, 1000, 10000, 100000, 10000000]
resultados_alpha = []

for a in alphas:
    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('model', linear_model.Ridge(alpha=a, random_state=42))
    ])

    resultados = cross_validate(
        pipeline, 
        X, 
        y, 
        cv=10,
        scoring=['r2', 'neg_mean_absolute_error', 'neg_root_mean_squared_error']
    )

    resultados_alpha.append({
        'alpha': a,
        'R2_médio': resultados['test_r2'].mean(),
        'MAE_médio': -resultados['test_neg_mean_absolute_error'].mean(),
        'RMSE_médio': -resultados['test_neg_root_mean_squared_error'].mean()
    })

results = pd.DataFrame(resultados_alpha)
print(results)

Os resultados mostram que nenhum valor de $\alpha$ produziu desempenho superior.

In [ ]:
alphas = [0.01, 0.1, 1, 10, 100, 1000, 10000, 100000, 10000000, 100000000]
ridge_cv = RidgeCV(alphas=alphas, scoring='neg_mean_squared_error')
ridge_cv.fit(X_train, y_train)
print("Melhor alpha (Ridge):", ridge_cv.alpha_)

Seleção automática do hiperparâmetro $\alpha$ no modelo Ridge via validação cruzada.

In [ ]:
model_ridge = linear_model.Ridge(alpha=100)
model_ridge.fit(X_train_scaled, y_train)

In [ ]:
y_pred_ridge_train = model_ridge.predict(X_train_scaled)
y_pred_ridge_test = model_ridge.predict(X_test_scaled)

r2_ridge_train = metrics.r2_score(y_train, y_pred_ridge_train)
r2_ridge_test = metrics.r2_score(y_test, y_pred_ridge_test)

mae_ridge_train = metrics.mean_absolute_error(y_train, y_pred_ridge_train)
mae_ridge_test = metrics.mean_absolute_error(y_test, y_pred_ridge_test)

rmse_ridge_train = np.sqrt(metrics.mean_squared_error(y_train, y_pred_ridge_train))
rmse_ridge_test = np.sqrt(metrics.mean_squared_error(y_test, y_pred_ridge_test))

print("RESULTADOS RIDGE (Treino vs Teste)")
print(f"R² Treino: {r2_ridge_train:.4f}")
print(f"R² Teste:  {r2_ridge_test:.4f}")
print(f"MAE Treino: {mae_ridge_train:,.2f}")
print(f"MAE Teste:  {mae_ridge_test:,.2f}")
print(f"RMSE Treino: {rmse_ridge_train:,.2f}")
print(f"RMSE Teste:  {rmse_ridge_test:,.2f}")

In [ ]:
print("Coeficientes do Modelo")
coef_ridge = pd.DataFrame(model_ridge.coef_, X.columns, columns=['Coeficiente'])
print(coef_ridge)

Mesmo com valor alto de $\alpha$ não há grandes mudanças ($\alpha = 100$). Isso significa que o problema não apresenta multicolinearidade forte nem sobreajuste relevante. O modelo $OLS$ já era suficientemente estável e generalizava bem aos dados. 

# LASSO

O modelo **Lasso** utiliza uma penalização L1, que além de reduzir *overfitting*, também zera alguns coeficientes, funcionando como um método de seleção automática de variáveis:

$$
\sum_{i=1}^{n} (y_i - \hat{y_i})^2 + \alpha \sum_{j=1}^{p} |\theta_j|
$$

O Lasso tende a produzir modelos mais simples e interpretáveis, mantendo apenas as variáveis mais relevantes.  

**Usando K-Folds**

Foram testados diversos valores do parâmetro de regularização $\alpha$ para o modelo Lasso. O objetivo era verificar se o acréscimo da penalização L1 melhoraria o desempenho do modelo em relação à Regressão Linear simples.

In [ ]:
# CORRIGIR R^2 NEGATIVO

alphas = [0.1, 1, 10, 100, 500, 800, 1000, 1500]
resultados_alpha = []

for a in alphas:
    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('model', linear_model.Lasso(alpha=a, random_state=42))
    ])

    resultados = cross_validate(
        pipeline,
        X,
        y,
        cv=10,
        scoring=['r2', 'neg_mean_absolute_error', 'neg_root_mean_squared_error']
    )

    resultados_alpha.append({
        'alpha': a,
        'R2_médio': resultados['test_r2'].mean(),
        'MAE_médio': -resultados['test_neg_mean_absolute_error'].mean(),
        'RMSE_médio': -resultados['test_neg_root_mean_squared_error'].mean()
    })

results = pd.DataFrame(resultados_alpha)
print(results)

In [ ]:
model_lasso = linear_model.Lasso(alpha=800)
model_lasso.fit(X_train_scaled, y_train)

In [ ]:
y_pred_lasso_train = model_lasso.predict(X_train_scaled)
y_pred_lasso_test = model_lasso.predict(X_test_scaled)

r2_lasso_train = metrics.r2_score(y_train, y_pred_lasso_train)
r2_lasso_test = metrics.r2_score(y_test, y_pred_lasso_test)

mae_lasso_train = metrics.mean_absolute_error(y_train, y_pred_lasso_train)
mae_lasso_test = metrics.mean_absolute_error(y_test, y_pred_lasso_test)

rmse_lasso_train = np.sqrt(metrics.mean_squared_error(y_train, y_pred_lasso_train))
rmse_lasso_test = np.sqrt(metrics.mean_squared_error(y_test, y_pred_lasso_test))

print("RESULTADOS LASSO (Treino vs Teste)")
print(f"R² Treino: {r2_lasso_train:.4f}")
print(f"R² Teste:  {r2_lasso_test:.4f}")
print(f"MAE Treino: {mae_lasso_train:,.2f}")
print(f"MAE Teste:  {mae_lasso_test:,.2f}")
print(f"RMSE Treino: {rmse_lasso_train:,.2f}")
print(f"RMSE Teste:  {rmse_lasso_test:,.2f}")

In [ ]:
print("Coeficientes do Modelo")
coef_lasso = pd.DataFrame(model_lasso.coef_, X.columns, columns=['Coeficiente'])
print(coef_lasso)

In [ ]:
print("Coeficientes do Modelo (≠ 0)")
coef_lasso = pd.DataFrame(model_lasso.coef_, index=X.columns, columns=['Coeficiente'])
coef_nao_zero = coef_lasso[coef_lasso['Coeficiente'] != 0]
print(coef_nao_zero)

O Lasso eliminou todas as variáveis menos relevantes, mantendo as que têm maior poder explicativo sobre o preço da passagem. Os resultados nos mostram que:



```
days_left        -1016.037932
airline_Vistara   1132.988774
stops_zero       -1969.728976
class_Economy   -19995.947955
```



O fator mais determinante é o preço. Passagens em classe econômica são muito mais baratas. 

Ter paradas reduz bastante o preço. A diferença principal está entre ter ou não paradas. O número de paradas adicionais tem impacto menor comparado a essa distinção. 


# Comparação Entre os Modelos Lineares

A seguir vamos comparar os resultados entre os modelos: OLS, Ridge e Lasso. 

In [ ]:
comparacao_resultados = pd.DataFrame({
    'Modelo': ['OLS', 'Ridge', 'Lasso'],
    'R²': [r2_test, r2_ridge_test, r2_lasso_test],
    'MAE': [mae_test, mae_ridge_test, mae_lasso_test],
    'RMSE': [rmse_test, rmse_ridge_test, rmse_lasso_test]
})

print("Comparação de Desempenho dos Modelos")
print(comparacao_resultados)

In [ ]:
coef_ols = pd.Series(model.coef_, index=X.columns)
coef_ridge = pd.Series(model_ridge.coef_, index=X.columns)
coef_lasso = pd.Series(model_lasso.coef_, index=X.columns)

comparacao_coef = pd.DataFrame({
    'OLS': coef_ols,
    'Ridge': coef_ridge,
    'Lasso': coef_lasso
})

print("Comparação dos Coeficientes dos Modelos")
print(comparacao_coef)

normas = pd.DataFrame({
    'Modelo': ['OLS', 'Ridge', 'Lasso'],
    'Norma L1 (∑|coef|)': [norm(coef_ols, 1), norm(coef_ridge, 1), norm(coef_lasso, 1)],
    'Norma L2 (||coef||²)': [norm(coef_ols, 2), norm(coef_ridge, 2), norm(coef_lasso, 2)]
})

print()

print("Comparação das Normas dos Coeficientes")
print(normas)

In [ ]:
print("Coeficientes do Lasso Diferentes de Zero (Variáveis Selecionadas)")
coef_lasso_df = pd.DataFrame(model_lasso.coef_, index=X.columns, columns=['Coeficiente'])
coef_lasso_ativo = coef_lasso_df[coef_lasso_df['Coeficiente'] != 0]
print(coef_lasso_ativo)


In [ ]:
model_names = ["OLS", "Ridge", "Lasso"]

r2_values   = [r2_test,       r2_ridge_test,   r2_lasso_test]
mae_values  = [mae_test,      mae_ridge_test,  mae_lasso_test]
rmse_values = [rmse_test,     rmse_ridge_test, rmse_lasso_test]

def plot_metric_with_zoom(values, title, ylabel, fmt="{:.4f}"):
    plt.figure()
    bars = plt.bar(model_names, values)
    plt.title(title)
    plt.ylabel(ylabel)

    min_v, max_v = min(values), max(values)
    if np.isclose(max_v, min_v):
        margin = abs(max_v) * 0.02 if abs(max_v) > 0 else 0.02
    else:
        margin = (max_v - min_v) * 0.2
    plt.ylim(min_v - margin, max_v + margin)

    for bar, v in zip(bars, values):
        plt.text(bar.get_x() + bar.get_width() / 2, v, fmt.format(v),
                 ha='center', va='bottom', fontsize=9)
    plt.grid(axis='y', alpha=0.2)
    plt.show()

plot_metric_with_zoom(r2_values, "Comparação de R² entre Modelos", "R²", "{:.4f}")
plot_metric_with_zoom(mae_values, "Comparação de MAE entre Modelos", "MAE", "{:.3f}")
plot_metric_with_zoom(rmse_values, "Comparação de RMSE entre Modelos", "RMSE", "{:.3f}")

coef_ols   = np.array(model.coef_)
coef_ridge = np.array(model_ridge.coef_)
coef_lasso = np.array(model_lasso.coef_)

feature_index = list(X.columns) if hasattr(X, "columns") else list(range(len(coef_ols)))

comparacao_coef = pd.DataFrame({
    "feature": feature_index,
    "OLS": coef_ols,
    "Ridge": coef_ridge,
    "Lasso": coef_lasso
})
display(comparacao_coef.head(20))

plt.figure(figsize=(10, 5))
plt.plot(coef_ols,   marker='o', linestyle='-', label='OLS')
plt.plot(coef_ridge, marker='o', linestyle='-', label='Ridge')
plt.plot(coef_lasso, marker='o', linestyle='-', label='Lasso')
plt.title("Coeficientes por Feature (ordem das features em X.columns)")
plt.xlabel("Índice da feature")
plt.ylabel("Valor do coeficiente")
plt.legend()
plt.grid(alpha=0.2)
plt.show()

norm_l1 = [np.sum(np.abs(coef_ols)), np.sum(np.abs(coef_ridge)), np.sum(np.abs(coef_lasso))]
norm_l2 = [np.sqrt(np.sum(coef_ols**2)), np.sqrt(np.sum(coef_ridge**2)), np.sqrt(np.sum(coef_lasso**2))]

plt.figure()
bars = plt.bar(model_names, norm_l1)
plt.title("Norma L1 dos Coeficientes")
plt.ylabel("L1")
for bar, v in zip(bars, norm_l1):
    plt.text(bar.get_x() + bar.get_width()/2, v, f"{v:.3f}", ha='center', va='bottom', fontsize=9)
plt.grid(axis='y', alpha=0.15)
plt.show()

plt.figure()
bars = plt.bar(model_names, norm_l2)
plt.title("Norma L2 dos Coeficientes")
plt.ylabel("L2")
for bar, v in zip(bars, norm_l2):
    plt.text(bar.get_x() + bar.get_width()/2, v, f"{v:.3f}", ha='center', va='bottom', fontsize=9)
plt.grid(axis='y', alpha=0.15)
plt.show()

n_nonzero = [np.sum(coef_ols != 0), np.sum(coef_ridge != 0), np.sum(coef_lasso != 0)]
plt.figure()
bars = plt.bar(model_names, n_nonzero)
plt.title("Número de coeficientes ≠ 0")
plt.ylabel("Nº coeficientes ≠ 0")
for bar, v in zip(bars, n_nonzero):
    plt.text(bar.get_x() + bar.get_width()/2, v, f"{int(v)}", ha='center', va='bottom', fontsize=9)
plt.grid(axis='y', alpha=0.15)
plt.show()


A regularização tem como principal objetivo controlar a complexidade do modelo e evitar overfitting, penalizando coeficientes muito grandes.  
Nos resultados obtidos:

- **Ridge Regression (penalização L2):**  
  O modelo Ridge produziu resultados praticamente idênticos ao OLS (R² ≈ 0.912), mostrando que a penalização foi leve, mas suficiente para reduzir ligeiramente a variância dos coeficientes.  

- **Lasso Regression (penalização L1):**  
  Essa simplificação vem com um pequeno custo de desempenho, já que o R² caiu levemente para ≈0.902.

A regularização L2 (Ridge) melhora a estabilidade sem perda de desempenho, enquanto a regularização L1 (Lasso) promove interpretabilidade ao custo de leve redução na precisão. 

# KNN

Agora faremos uma nova regressão baseado em ideias diferentes que é o K-nearest neighbours (KNN). Esse método utiliza os K vizinhos mais próximos de um ponto para determinar seu valor. O número K é um hiperparametro, nós precisamos usar métodos para descobrir seu valor que melhor se encaixe ao nosso problema e nesse caso faremos isso através de cross-validation para evitar overfitting do hiperparametro

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import GridSearchCV
from sklearn import metrics


# 1) Pipeline
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('knn', KNeighborsRegressor())
])

# 2) Search space
param_grid = {
    'knn__n_neighbors': [3,5,7,9,11,13,15],
}

# 3) GridSearch com cross validation
grid = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=5,
    scoring='r2',
    n_jobs=-1
)

# 4) treina só com train
grid.fit(X_train, y_train)

print("Melhores hiperparâmetros encontrados:")
print(grid.best_params_)

# 5) melhor modelo já vem internamente
best_model = grid.best_estimator_

# 6) avaliação final no TEST
y_pred = best_model.predict(X_test)

print("\n--- Resultados finais no Test ---")
r2 = metrics.r2_score(y_test, y_pred)
print(f"R2 final: {r2:.4f}")


Aqui notamos que o modelo que se saiu melhor foi o com K = 3. Nesse caso testamos apenas para alguns valores de K pois é inviável testar para todos os valores possíveis de K logo testamos os mais razoáveis e comuns. Percebmos que esse modelo obteve um ótimo o que indica que podemos tentar apostar em modelos de árvores que tendem a ser melhores em generalizar mas mantém uma lógica parecida com O KNN já que ambos são modelos não paramétricos, em contraste com os outros modelos de aprendizado supervisionado treinados até aqui que são paramétricos

# GAM 


O **Modelo Aditivo Generalizado (GAM)** é uma extensão da regressão linear que permite capturar relações não lineares. 

O modelo pode ser escrito da seguinte forma:

$$
y=\beta_0 + f_1(x_1) +...+ f_p(x_p) + ϵ
$$

onde $f_i(x_i)$ é função suave que modela o efeito da variável. 

In [ ]:
from sklearn.preprocessing import LabelEncoder

cat_cols = ['airline','source_city','departure_time','stops','arrival_time','destination_city','class','cluster']
for c in cat_cols:
    le = LabelEncoder()
    df_flight[c] = le.fit_transform(df_flight[c])

X = df_flight[['airline','source_city','departure_time','stops','arrival_time',
        'destination_city','class','duration','days_left','cluster']]
y = df_flight['price']

X_train_GAM, X_test_GAM, y_train_GAM, y_test_GAM = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [ ]:
gam_model = LinearGAM(
    s(0) +        # duration
    s(1) +        # days_left
    f(2) +        # airline
    f(3) +        # source_city
    f(4) +        # departure_time
    f(5) +        # stops
    f(6) +        # arrival_time
    f(7) +        # destination_city
    f(8) +        # class
    f(9)          # cluster
).fit(X_train_GAM, y_train_GAM)

print(gam_model.summary())
y_pred_gam = gam_model.predict(X_test_GAM)


In [ ]:
print("\n--- Resultados da Avaliação do Modelo GAM (Smoothing Splines) ---")

# R-quadrado (R²)
r2_gam = metrics.r2_score(y_test_GAM, y_pred_gam)
print(f"R-quadrado (R²): {r2_gam:.4f}")

In [ ]:
mae_gam = metrics.mean_absolute_error(y_test_GAM, y_pred_gam)
print(f"Erro Absoluto Médio (MAE): ${mae_gam:,.2f}")

In [ ]:
rmse_gam = np.sqrt(metrics.mean_squared_error(y_test_GAM, y_pred_gam))
print(f"Raiz do Erro Quadrático Médio (RMSE): ${rmse_gam:,.2f}")

# XGBOOST


Agora, baseado no bom desempenho do KNN tentaremos ajustar modelos baseados em arvores de decisão, começando pelo XGBOost que é um método como o nome diz de boosting. Métodos de boosting tentam treinam árvores fracas que "consertam" os erros das árvores anteriores, basicamente começando com uma árvore de alto viés e tentando adicionar variância até chegar no ponto ideal do trade-off. O XGBoost especificamente funciona utilizando métodos de gradiente descendente para mimizar o erro de forma eficiente que da o nome ao G do método e o X vem de diversar otimizações em relação a outros métodos que utilizam gradiente para boosting, como regularização, otimização de hardwere e inúmeros hiperparametro que possibilitam fine tunning do modelo.

In [ ]:
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.metrics import r2_score, make_scorer
import xgboost as xgb

# --- Modelo base com GPU ativada ---
xgb_model = xgb.XGBRegressor(
    objective='reg:squarederror',
    random_state=42,
    n_jobs=-1,
    tree_method='gpu_hist',       # <--- usa GPU no treino
    predictor='gpu_predictor'     # <--- usa GPU na predição
)

# --- Grid de hiperparâmetros ---
param_grid = {
    'n_estimators': [200, 400, 600],
    'learning_rate': [0.01, 0.05, 0.1],
    'max_depth': [3, 5, 7],
    'min_child_weight': [1, 3, 5],
    'subsample': [0.7, 0.9, 1.0],
    'colsample_bytree': [0.7, 0.9, 1.0],
    'reg_lambda': [1.0, 1.5, 2.0],
    'reg_alpha': [0, 0.1, 0.5]
}

kf = KFold(n_splits=5, shuffle=True, random_state=42)
r2_scorer = make_scorer(r2_score)

grid_search = GridSearchCV(
    estimator=xgb_model,
    param_grid=param_grid,
    scoring=r2_scorer,
    cv=kf,
    verbose=2,
    n_jobs=-1
)

# --- Treinamento ---
grid_search.fit(X, y)

print("\n======== RESULTADO GRID SEARCH (GPU) ========")
print("Melhores hiperparâmetros:")
for param, value in grid_search.best_params_.items():
    print(f"{param}: {value}")
print("------------------------------------------")
print(f"Melhor R² médio (CV): {grid_search.best_score_:.4f}")

best_model = grid_search.best_estimator_


Reparamos aqui que esse método se saiu muito bem superando ainda por um pequeno valor o KNN.

# RANDOM FOREST

Aqui utilizamoso  random forest, um método também baseado em árvores mas dessa vez em bagging, ele cria uma floresta com quantidade de árvores desejadas utilizando um fator aleátorio para decidir quais predtores pode utilizar em cada nó da árvore, assim ao fazera a média de todas as árvores da floresta a alta variancia de cada uma diminue drasticamente e assim temos um modelo mais robusto.

In [ ]:
from sklearn.model_selection import KFold, cross_val_score
from sklearn.metrics import r2_score, make_scorer
from sklearn.ensemble import RandomForestRegressor

rf_model = RandomForestRegressor(
    n_estimators=200,
    random_state=42
)

kf = KFold(n_splits=5, shuffle=True, random_state=42)

r2_scorer = make_scorer(r2_score)

results = cross_val_score(rf_model, X, y, cv=kf, scoring=r2_scorer)

print("======== RESULTADO K-FOLD RANDOM FOREST ========")
print("Scores por Fold (R²):")
for i, r in enumerate(results):
    print(f"Fold {i+1}: {r:.4f}")

print("------------------------------------------")
print(f"Média R²: {results.mean():.4f}")
print(f"Desvio Padrão: {results.std():.4f}")


In [ ]:
from cuml.ensemble import RandomForestRegressor as cuRF
from sklearn.model_selection import KFold, GridSearchCV
from sklearn.metrics import r2_score, make_scorer

rf_gpu = cuRF(random_state=42, n_streams=8)  # usa CUDA!

param_grid = {
    'n_estimators': [100, 200, 400],
    'max_depth': [10, 20, 30],
    'max_features': ['auto', 1.0],
    'min_samples_split': [2, 5, 10],
}

kf = KFold(n_splits=5, shuffle=True, random_state=42)
r2_scorer = make_scorer(r2_score)

grid_search = GridSearchCV(
    estimator=rf_gpu,
    param_grid=param_grid,
    scoring=r2_scorer,
    cv=kf,
    verbose=2
)

grid_search.fit(X, y)

print("\n======== RESULTADO GRID SEARCH RANDOM FOREST (GPU) ========")
print("Melhores hiperparâmetros:")
print(grid_search.best_params_)
print("------------------------------------------")
print(f"Melhor R² médio (CV): {grid_search.best_score_:.4f}")


In [ ]:
rf_model.fit(X, y)
importances = pd.Series(rf_model.feature_importances_, index=X.columns)
importances.sort_values(ascending=False).plot(kind='bar', figsize=(10,5))
plt.title("Importância das Variáveis na Random Forest")
plt.show()


# REDES NEURAIS


Por fim, ajustaremos uma rede neural, um dos tipos de modelo mais famosos atualmente. Nossa rede neural que tenta descrever a relação dos preditores com o preço é de uma arquitetura que contém3 camadas ocultas com 64, 32 e 16 neuronios respectivamente e completamente conectada. Utilizamos a função ReLU que é uma das funções de ativação mais populares, uma taxa de aprendizado de 0.001 contante durante todo o treinament que também é um valor comum, 1000 passagens pelos dados, o solver adam que é um dos mais robustos. ALém disso, estamos usando um fator de regularização alpha de 0.01 que é a constante multiplicativa do termo da soma das normas L2 de cada parametro na função de perda, ou seja, alem de minimizar o erro minmiazmos a soma das normas de cada parametro e também utilizamos o early_stopping que para o treino se a perfromance nao melhorar.

In [ ]:
mlp_model = MLPRegressor(
    hidden_layer_sizes=(64, 32, 16), # Duas camadas ocultas: a primeira com 64 neurônios, a segunda com 32.
    activation='relu',           # Função de ativação padrão e mais eficiente.
    solver='adam',               # O otimizador mais popular e robusto.
    max_iter=1000,               # Número máximo de épocas (passagens pelos dados).
    alpha=0.01,                  # Parâmetro de regularização para evitar overfitting.
    early_stopping=True,         # Para o treino se a performance não melhorar, evitando overfitting.
    random_state=42
)

# Treinar o modelo com os dados padronizados
mlp_model.fit(X_train_scaled, y_train)

print("Treinamento concluído!")

# --- 4. Avaliação ---
# Fazer previsões com os dados de teste PADRONIZADOS
y_pred_mlp = mlp_model.predict(X_test_scaled)

# Calcular o R²
r2_mlp = metrics.r2_score(y_test, y_pred_mlp)

# Supondo que você salvou o r2 da regressão linear na variável r2_linear
print("\n--- RESULTADO COM MLP (Rede Neural) ---")
print(f"R² do MLP Regressor: {r2_mlp:.4f}")